# DE/PARA: Cursos FATEC × Ocupações CBO (2025)

## Objetivo
Mapear cada curso da FATEC para a ocupação CBO mais relevante com base em similaridade semântica, usando como base as ocupações mais frequentes nas movimentações do CAGED em 2025.

## Metodologia
1. Identifica as 60 ocupações CBO mais frequentes no CAGED 2025 (filtradas por descritores técnicos: analista, programador, administrador, designer, técnico, engenheiro, etc.)
2. Calcula a similaridade semântica entre cada curso FATEC (+ eixo tecnológico) e cada ocupação CBO usando `ai_similarity()`
3. Seleciona a ocupação com maior similaridade para cada curso
4. Arredonda o score de similaridade para 3 casas decimais
5. Classifica como match "forte" (≥ 0.7) ou "fraco" (< 0.7)

## Resultado
Tabela `workspace.silver.cursos_fatec_ocupacoes_cbo_2025` com 116 linhas (um curso por linha) e 8 colunas.


## Passo 1: Preparar os candidatos de ocupação

Identifica as 60 ocupações CBO mais frequentes no CAGED 2025 que atendem critérios técnicos.

In [ ]:
-- Identifica as 60 ocupações CBO mais frequentes no CAGED 2025
-- Filtrada por descritores de ocupações técnicas/profissionais
WITH top_ocup AS (
  SELECT m.`cbo2002ocupação` AS codigo, count(*) AS mov
  FROM workspace.bronze.novo_caged_movimentacoes m
  WHERE m.`_competencia_referencia` LIKE '2025%'
  GROUP BY 1
)
SELECT o.codigo, o.descricao, t.mov
FROM workspace.bronze.ref_cbo_ocupacao o
JOIN top_ocup t ON t.codigo = o.codigo
WHERE o.descricao RLIKE '(?i)(analista|programador|administrador|designer|desenhista|tecnologo|tecnico|engenheiro|cientista|agronomo|logistic|marketing|publicidade|jornalist|moda)'
ORDER BY t.mov DESC
LIMIT 60;

## Passo 2: Executar o cruzamento semântico completo

Calcula similaridade entre cada curso e cada ocupação candidata, seleciona o melhor match e grava em Silver.

In [ ]:
-- Cruzamento: cursos FATEC x ocupacoes CBO (base: movimentacoes CAGED 2025)
-- Similaridade semantica (ai_similarity) entre Curso + Eixo Tecnologico e a
-- descricao da ocupacao CBO; candidatos = 60 ocupacoes qualificadas mais
-- frequentes no CAGED 2025. Salva a melhor ocupacao por curso.
CREATE OR REPLACE TABLE workspace.silver.cursos_fatec_ocupacoes_cbo_2025 AS
WITH cand AS (
  WITH top_ocup AS (
    SELECT m.`cbo2002ocupação` AS codigo, count(*) AS mov
    FROM workspace.bronze.novo_caged_movimentacoes m
    WHERE m.`_competencia_referencia` LIKE '2025%'
    GROUP BY 1
  )
  SELECT o.codigo, o.descricao, t.mov
  FROM workspace.bronze.ref_cbo_ocupacao o
  JOIN top_ocup t ON t.codigo = o.codigo
  WHERE o.descricao RLIKE '(?i)(analista|programador|administrador|designer|desenhista|tecnologo|tecnico|engenheiro|cientista|agronomo|logistic|marketing|publicidade|jornalist|moda)'
  ORDER BY t.mov DESC
  LIMIT 60
),
matching AS (
  SELECT trim(c.Curso) AS curso, trim(c.eixo) AS eixo_tecnologico, cd.codigo AS cbo_codigo,
         cd.descricao AS ocupacao_cbo, cd.mov AS movimentacoes_2025,
         ai_similarity(concat(trim(c.Curso), coalesce(concat(' - ', trim(c.eixo)), '')), cd.descricao) AS similaridade
  FROM (
    SELECT Curso, `Eixo Tecnológico` AS eixo
    FROM workspace.bronze.cursos_fatec
  ) c
  CROSS JOIN cand cd
),
ranked AS (
  SELECT *, row_number() OVER (PARTITION BY curso ORDER BY similaridade DESC) AS rn
  FROM matching
)
SELECT curso, eixo_tecnologico, ocupacao_cbo, cbo_codigo, movimentacoes_2025, 
       round(similaridade, 3) AS similaridade,
       CASE WHEN round(similaridade, 3) >= 0.7 THEN 'forte' ELSE 'fraco' END AS forca_match
FROM ranked
WHERE rn = 1;

## Passo 3: Validação e exploração

Verifica o resultado final.

In [ ]:
-- Contagem total e distribuição de força de match
SELECT forca_match, COUNT(*) AS cursos, COUNT(*) * 100.0 / 116 AS pct
FROM workspace.silver.cursos_fatec_ocupacoes_cbo_2025
GROUP BY forca_match
ORDER BY forca_match DESC;

In [ ]:
-- Preview das primeiras linhas
SELECT * FROM workspace.silver.cursos_fatec_ocupacoes_cbo_2025 LIMIT 10;

In [ ]:
-- Distribuição de similaridade
SELECT 
  COUNT(*) AS total,
  MIN(similaridade) AS min_sim,
  MAX(similaridade) AS max_sim,
  ROUND(AVG(similaridade), 3) AS media_sim,
  PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY similaridade) AS mediana_sim
FROM workspace.silver.cursos_fatec_ocupacoes_cbo_2025;